# Emote: train the emotion + sarcasm models on a Colab GPU

**Before running:** Runtime > Change runtime type > **T4 GPU**.

On your PC, zip the `ml` folder **without** `.venv`, `model` and `__pycache__` (the `.py` files, `requirements.txt` is enough), then upload it in step 2.

Expected time on a free T4: roughly 1-1.5 h for `roberta-large`, 20-30 min for `roberta-base`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: switch the runtime to T4 GPU first'
print(torch.cuda.get_device_name(0))
!pip -q install 'transformers>=4.46' datasets accelerate scikit-learn scipy

In [ ]:
# 2. Upload ml.zip (the ml folder without .venv / model)
from google.colab import files
up = files.upload()
!unzip -oq ml.zip -d ml_src
%cd ml_src
!ls

In [ ]:
# 3. Settings. roberta-large is the most accurate; use roberta-base if it runs out of memory
# or you want faster inference in the app.
BASE = 'roberta-large'

In [ ]:
# 4. Emotion model (GoEmotions, 28 labels, per-class thresholds)
!python train.py --base $BASE --epochs 4 --batch-size 32 --out model

In [ ]:
# 5. Sarcasm model (iSarcasmEval + tweet_eval irony + Reddit)
!python train_sarcasm.py --base $BASE --epochs 3 --out sarcasm_model

In [ ]:
# 6. Full evaluation report (paper's metrics: F1, MAE, correlation, calibration...)
!python evaluate.py --model model --sarcasm sarcasm_model

In [ ]:
import os
from google.colab import files

# Results first: paste this output back into the chat
for d in ('model', 'sarcasm_model'):
    p = f'{d}/metrics.json'
    if os.path.exists(p):
        print(f'===== {p} =====')
        print(open(p).read())

# Zip whichever models exist and download them directly (no Google Drive needed).
dirs = [d for d in ('model', 'sarcasm_model') if os.path.isdir(d)]
print('Zipping:', dirs)
!zip -qr trained_models.zip {' '.join(dirs)}
!ls -lh trained_models.zip
files.download('trained_models.zip')
# If the browser stalls on a multi-GB file, download each folder separately:
#   !zip -qr model.zip model;                 files.download('model.zip')
#   !zip -qr sarcasm_model.zip sarcasm_model; files.download('sarcasm_model.zip')
